# Scaling Strategies (ANN)

## Key Scaling Considerations

1. **Speed vs. Accuracy** - Understanding the tradeoffs between query performance and result quality
2. **Resource Limitations** - Managing memory, CPU, and storage constraints
3. **Horizontal Scaling** - Distributing the workload across multiple instances

## Approximate Nearest Neighbor (ANN) Implementations

ANN algorithms like HNSW (Hierarchical Navigable Small World) allow us to trade some accuracy for significant performance improvements at scale. We'll explore different HNSW configurations and their impact on search performance.

# Setup and Initialization

In [7]:
# Install the required packages
!uv pip install accelerate==1.6.0 sentence-transformers==4.0.2

'uv' is not recognized as an internal or external command,
operable program or batch file.


In [8]:
import chromadb
from chromadb.utils import embedding_functions
import time

# Initialize ChromaDB client
client = chromadb.Client()
embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
  model_name="all-MiniLM-L6-v2"
)

### Creating Collections with Different HNSW Configurations

We'll create three collections with different index settings:

1. **Default** - Uses ChromaDB's default configuration
2. **High Accuracy** - Prioritizes result quality with higher `ef` and `M` values
3. **Fast Search** - Prioritizes speed with lower `ef` and `M` values

**Parameter Explanation:**
- `hnsw:space`: The distance metric used (cosine, euclidean, etc.)
- `hnsw:construction_ef`: Controls index build quality (higher = better quality, slower build)
- `hnsw:search_ef`: Controls search quality (higher = better quality, slower search)
- `hnsw:M`: Controls the maximum number of connections per node (higher = better quality, more memory)

In [9]:
# Create collections with different HNSW configurations
collections = {}

# 1. Default settings
collections["default"] = client.create_collection(
    name="default_index",
    embedding_function=embedding_function
)

# 2. High accuracy configuration
collections["high_accuracy"] = client.create_collection(
    name="high_accuracy_index",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine", "hnsw:construction_ef": 1000, "hnsw:search_ef": 1250, "hnsw:M": 36}
)

# 3. Fast search configuration
collections["fast_search"] = client.create_collection(
    name="fast_search_index",
    embedding_function=embedding_function,
    metadata={"hnsw:space": "cosine", "hnsw:construction_ef": 80, "hnsw:search_ef": 40, "hnsw:M": 12}
)

### Generating Sample Documents

Now let's create some sample documents across different categories to populate our collections.

In [10]:
# Generate sample data
num_docs = 10000
print(f"Generating {num_docs} sample documents...")

# Create documents with some patterns for testing
categories = ["technology", "science", "health", "business", "entertainment"]
documents = []
ids = []

for i in range(num_docs):
    category = categories[i % len(categories)]
    document = f"This is document {i} about {category} with some additional text to make it more unique."
    documents.append(document)
    ids.append(f"doc_{i}")

print(documents[0:5])

Generating 10000 sample documents...
['This is document 0 about technology with some additional text to make it more unique.', 'This is document 1 about science with some additional text to make it more unique.', 'This is document 2 about health with some additional text to make it more unique.', 'This is document 3 about business with some additional text to make it more unique.', 'This is document 4 about entertainment with some additional text to make it more unique.']


### Adding Documents to Collections

Let's add the generated documents to all three collections.

In [12]:
import time

print("Adding documents to collections with different index configurations...")

# ChromaDB has a maximum batch size, so we need to split into smaller batches
batch_size = 5000  # Safe batch size (under the 5461 limit)

for name, collection in collections.items():
    start_time = time.time()

    # Add documents in batches
    for i in range(0, len(documents), batch_size):
        batch_docs = documents[i:i + batch_size]
        batch_ids = ids[i:i + batch_size]
        collection.add(
            documents=batch_docs,
            ids=batch_ids
        )

    end_time = time.time()
    elapsed_time = end_time - start_time

    print(
        f"  Added {num_docs} documents to {name} collection in {elapsed_time:.4f} seconds")

Adding documents to collections with different index configurations...
  Added 10000 documents to default collection in 28.9585 seconds
  Added 10000 documents to high_accuracy collection in 29.5245 seconds
  Added 10000 documents to fast_search collection in 28.4102 seconds


### Benchmark Query Performance

Now let's evaluate how each configuration performs with a set of representative queries.

In [13]:
# Benchmark query performance
print("\nBenchmarking query performance across different configurations...")

# Prepare queries
query_texts = [
    "Latest technology trends in artificial intelligence",
    "Scientific research on climate change",
    "Health benefits of regular exercise",
    "Business strategies for startups",
    "Entertainment news about recent movie releases"
]

# Set up benchmark parameters
results = {}
num_trials = 5


Benchmarking query performance across different configurations...


In [14]:
# Run benchmark for each collection
for name, collection in collections.items():
    print(f"\nTesting {name} configuration:")
    times = []
    
    for query in query_texts:
        query_times = []
        
        for _ in range(num_trials):
            start_time = time.time()
            collection.query(
                query_texts=[query],
                n_results=10
            )
            query_time = time.time() - start_time
            query_times.append(query_time)
        
        avg_time = sum(query_times) / len(query_times)
        times.append(avg_time)
        print(f"  Query: '{query[:30]}...': {avg_time:.4f} seconds")
    
    results[name] = {
        "mean": sum(times) / len(times),
        "min": min(times),
        "max": max(times),
        "times": times
    }


Testing default configuration:
  Query: 'Latest technology trends in ar...': 0.0495 seconds
  Query: 'Scientific research on climate...': 0.0220 seconds
  Query: 'Health benefits of regular exe...': 0.0187 seconds
  Query: 'Business strategies for startu...': 0.0181 seconds
  Query: 'Entertainment news about recen...': 0.0171 seconds

Testing high_accuracy configuration:
  Query: 'Latest technology trends in ar...': 0.0265 seconds
  Query: 'Scientific research on climate...': 0.0204 seconds
  Query: 'Health benefits of regular exe...': 0.0190 seconds
  Query: 'Business strategies for startu...': 0.0195 seconds
  Query: 'Entertainment news about recen...': 0.0187 seconds

Testing fast_search configuration:
  Query: 'Latest technology trends in ar...': 0.0259 seconds
  Query: 'Scientific research on climate...': 0.0177 seconds
  Query: 'Health benefits of regular exe...': 0.0173 seconds
  Query: 'Business strategies for startu...': 0.0176 seconds
  Query: 'Entertainment news about recen

In [15]:
# Print summary of benchmark results
print("\nPerformance Summary:")
for name, metrics in results.items():
    print(f"  {name}: Mean={metrics['mean']:.4f}s, Min={metrics['min']:.4f}s, Max={metrics['max']:.4f}s")


Performance Summary:
  default: Mean=0.0251s, Min=0.0171s, Max=0.0495s
  high_accuracy: Mean=0.0208s, Min=0.0187s, Max=0.0265s
  fast_search: Mean=0.0192s, Min=0.0173s, Max=0.0259s


## Conclusion and Key Takeaways

In this notebook, we've explored practical approaches to scaling vector databases for production use using ANNs:

**ANN Implementations**
   - Configuring HNSW parameters allows for customized speed-accuracy tradeoffs
   - The right configuration depends on your specific application requirements